# D191 — Python Virtual Environment and Jupyter on WSL

This hands-on lab creates a Python 3.14 virtual environment named **dataengenv** in the WSL user's home directory. It covers activation, deactivation, pandas, Jupyter, automatic Bash activation, and a compact prompt.

> Run commands in Ubuntu/WSL Bash, not PowerShell. Examples use `$HOME` and `$USER`; no account name or password is stored.

## 1. Verify Python and install venv support

```bash
echo "User: $USER"
echo "Home: $HOME"
echo "Shell: $SHELL"
python3 --version
command -v python3
python3 -c 'import sys; print(sys.executable); print(sys.version)'
python3 -m venv --help | head
```

The class machine should report Python 3.14. If the final command reports missing `venv` or `ensurepip`, install support:

```bash
sudo apt update
sudo apt install python3.14-venv
```

If the configured Ubuntu repository provides only its default package, use `sudo apt install python3-venv`. Enter any sudo password only at the interactive prompt; never store it in a command or notebook.

## 2. Create dataengenv in the home directory

Do not use sudo. The environment must belong to the normal WSL user.

```bash
python3 -m venv "$HOME/dataengenv"
ls -la "$HOME/dataengenv"
ls -la "$HOME/dataengenv/bin" | head
```

The environment contains its own Python launcher, pip, activation scripts, and package directory. It does not replace the operating system's Python.

## 3. Activate, inspect, deactivate, and reactivate

Activation places the environment's `bin` directory first in `PATH` for the current shell:

```bash
source "$HOME/dataengenv/bin/activate"
echo "VIRTUAL_ENV=$VIRTUAL_ENV"
command -v python
command -v pip
python --version
python -c 'import sys; print(sys.executable); print(sys.prefix)'
```

The executable should be `$HOME/dataengenv/bin/python`. The POSIX spelling `. "$HOME/dataengenv/bin/activate"` is equivalent.

Deactivate:

```bash
deactivate
echo "VIRTUAL_ENV=${VIRTUAL_ENV:-not active}"
command -v python3
```

Reactivate for the rest of the lab:

```bash
source "$HOME/dataengenv/bin/activate"
echo "Active environment: $VIRTUAL_ENV"
```

Never use `sudo pip install`: it may bypass the environment and modify system-managed Python.

## 4. Upgrade pip and install pandas

Use `python -m pip` so pip certainly belongs to the selected interpreter.

```bash
python -m pip install --upgrade pip setuptools wheel
python -m pip --version
python -m pip install pandas
python -m pip show pandas
python -c 'import pandas as pd; print("pandas", pd.__version__)'
```

Demonstration:

```bash
python - <<'PY'
import pandas as pd

sales = pd.DataFrame({
    "product": ["book", "pen", "book"],
    "amount": [300, 50, 250],
})
print(sales)
print(sales.groupby("product", as_index=False)["amount"].sum())
PY
```

## 5. Install and start Jupyter

Install Jupyter while dataengenv is active:

```bash
python -m pip install jupyter
command -v jupyter
jupyter --version
python -m pip show jupyter
```

The executable should be `$HOME/dataengenv/bin/jupyter`.

Create a workspace and start JupyterLab:

```bash
mkdir -p "$HOME/dataeng/notebooks"
cd "$HOME/dataeng/notebooks"
jupyter lab
```

WSL normally opens the Windows browser. Otherwise copy the complete localhost URL, including its token, from the terminal. Keep the terminal running. Stop the server with `Ctrl+C` and confirm.

Classic Notebook and useful alternatives:

```bash
jupyter notebook
jupyter lab --no-browser
jupyter lab --ServerApp.port=8889
jupyter lab --ServerApp.ip=127.0.0.1
jupyter server list
python -m jupyterlab
```

Keep the server bound to `127.0.0.1`. Do not expose it on all interfaces, disable authentication, or share token URLs without understanding the security impact.

## 6. Back up .bashrc

Bash reads `$HOME/.bashrc` for an interactive WSL Bash shell. Back it up before changing startup behavior:

```bash
cp -a "$HOME/.bashrc" "$HOME/.bashrc.backup.$(date +%Y%m%d_%H%M%S)"
ls -lt "$HOME"/.bashrc* | head
tail -n 25 "$HOME/.bashrc"
```

The timestamp preserves earlier backups.

## 7. Activate dataengenv automatically

Append this guarded block once. It activates only when the environment exists and another environment is not already active.

```bash
cat >> "$HOME/.bashrc" <<'EOF'

# --- dataengenv: automatic activation ---
if [ -z "${VIRTUAL_ENV:-}" ] && [ -f "$HOME/dataengenv/bin/activate" ]; then
    . "$HOME/dataengenv/bin/activate"
fi
# --- end dataengenv activation ---
EOF
```

The quoted `EOF` prevents expansion while writing. Variables are evaluated when a future shell reads `.bashrc`. Before repeating this command, avoid duplicate blocks by checking:

```bash
grep -n 'dataengenv' "$HOME/.bashrc"
```

## 8. Change the prompt to `$` followed by the path

- PS1 means “Primary Prompt String.”
- \$ displays $ for a normal user and # for root.
- \w displays the current working directory.

-  PS1='\$ \w '

Try to echo

Bash uses `PS1` for its prompt. `\u` is username, `\h` hostname, `\w` the current path with home abbreviated as `~`, `\W` the final directory only, and `\$` displays `$` for a normal user or `#` for root.

Append this **after** the activation block so it hides `username@hostname` and the usual `(dataengenv)` prefix:

```bash
cat >> "$HOME/.bashrc" <<'EOF'

# --- compact prompt: symbol followed by current directory ---
PS1='\$ \w '
# --- end compact prompt ---
EOF
```

Example:

```text
$ ~/dataeng/notebooks 
```

The virtual environment remains active even though its name is hidden. Check it with `echo "$VIRTUAL_ENV"`. Using `\$` rather than a literal dollar safely shows `#` inside a root shell.

For an even shorter prompt showing only the current directory's final component, use `PS1='\$ \W '` instead.

## 9. Validate and apply Bash changes

Check syntax before loading the file:

```bash
bash -n "$HOME/.bashrc" && echo '.bashrc syntax is valid'
source "$HOME/.bashrc"
```

Verify:

```bash
echo "VIRTUAL_ENV=$VIRTUAL_ENV"
command -v python
python --version
printf 'PS1=%q\n' "$PS1"
cd "$HOME/dataeng/notebooks"
```

Open a second Ubuntu/WSL terminal. It should activate `$HOME/dataengenv` automatically and display a prompt such as `$ ~/dataeng/notebooks`.

Auto-activation occurs when `.bashrc` is read. You can still deactivate for the current session:

```bash
deactivate
echo "${VIRTUAL_ENV:-No virtual environment is active}"
```

It activates again in a new terminal. To reactivate immediately, run `source "$HOME/dataengenv/bin/activate"`.

## 10. Troubleshoot or undo the startup changes

Start Bash without reading `.bashrc`:

```bash
bash --norc
```

Type `exit` to return. To disable auto-activation or the custom prompt, open `nano "$HOME/.bashrc"` and remove only the appropriate marked block. In nano, `Ctrl+W` searches, `Ctrl+O` saves, and `Ctrl+X` exits.

Restore a backup by selecting its exact name:

```bash
ls -lt "$HOME"/.bashrc.backup.*
cp "$HOME/.bashrc.backup.YYYYMMDD_HHMMSS" "$HOME/.bashrc"
bash -n "$HOME/.bashrc" && source "$HOME/.bashrc"
```

Replace the example timestamp. Do not use a broad wildcard as the `cp` source.

## 11. Record packages and verify the completed setup

```bash
mkdir -p "$HOME/dataeng"
python -m pip freeze > "$HOME/dataeng/requirements.txt"
head "$HOME/dataeng/requirements.txt"
```

Run final checks in a newly opened terminal:

```bash
test -d "$HOME/dataengenv" && echo 'environment directory exists'
test "${VIRTUAL_ENV:-}" = "$HOME/dataengenv" && echo 'environment is active'
test "$(command -v python)" = "$HOME/dataengenv/bin/python" && echo 'correct Python selected'
python --version
python -m pip --version
python -c 'import pandas; print("pandas", pandas.__version__)'
jupyter --version
jupyter server list
```

Do not copy or commit the virtual-environment directory. Recreate it with `python3 -m venv` and reinstall packages from a requirements file.

## Command summary

```bash
# Create
python3 -m venv "$HOME/dataengenv"

# Activate
source "$HOME/dataengenv/bin/activate"

# Deactivate
deactivate

# Install packages while active
python -m pip install pandas jupyter

# Start Jupyter
cd "$HOME/dataeng/notebooks"
jupyter lab

# Confirm interpreter
python -c 'import sys; print(sys.executable)'
```